# Agentic Artificial Intelligence
## Exercise - Unit 08: Augmented Generation (RAG & CAG)

Welcome to the eighth unit of the Agentic Artificial Intelligence course!

## Learning Objectives
By the end of this lesson, students will:
1. Understand what Augmented Generation is and why it's essential for LLM applications
2. Learn the fundamentals of Retrieval-Augmented Generation (RAG)
3. Learn the fundamentals of Cache-Augmented Generation (CAG)
4. Understand the differences, advantages, and limitations of RAG vs CAG
5. Implement a RAG system using vector stores and embeddings
6. Implement a CAG system by preloading context into LLM prompts
7. Understand when to use each approach in different scenarios
8. Build agents that leverage both RAG and CAG techniques

## Prerequisites
- Students should have completed Unit 05 exercises on tool integration
- Students should have completed Unit 06 exercises on MCP
- Students should have completed Unit 07 exercises on tracing
- Understanding of LangGraph basics (Unit 03)
- Familiarity with embeddings and vector stores (concepts from Unit 06)
- Basic Python knowledge (classes, functions, data structures)

# Update dependencies

As I added new packages, you must first run `uv sync` in order to run the code.

In [ ]:
!uv sync

If you face import issues run `uv sync` in the terminal from the root dir of this project.

## 1. What is Augmented Generation?

**Augmented Generation** refers to techniques that enhance Large Language Models (LLMs) by providing them with additional context or information beyond their training data. This allows LLMs to:

- **Access up-to-date information** - Information that wasn't in the training data
- **Ground responses in external data** - Reduce hallucinations by citing sources
- **Handle domain-specific knowledge** - Access specialized information
- **Improve accuracy** - Use verified, relevant information

### Why Do We Need Augmented Generation?

LLMs have several limitations:

1. **Static Training Data** - They're trained on data up to a certain date
2. **No Real-Time Information** - Can't access current events or recent data
3. **Limited Context Window** - Can't hold all knowledge in memory
4. **Hallucination Risk** - May generate plausible but incorrect information
5. **Domain Limitations** - May lack specialized knowledge

Augmented Generation solves these problems by:
- **Retrieving** relevant information when needed (RAG)
- **Preloading** relevant information into context (CAG)

### Two Main Approaches

In this unit, we'll explore two primary approaches:

1. **RAG (Retrieval-Augmented Generation)** - Retrieves relevant documents during inference
2. **CAG (Cache-Augmented Generation)** - Preloads relevant information into the model's context

Let's dive into each approach!

## 2. Retrieval-Augmented Generation (RAG)

**Retrieval-Augmented Generation (RAG)** is a technique that enhances LLM responses by retrieving relevant documents from an external knowledge base during inference and including them in the context.

### 2.1 How RAG Works

The RAG process follows these steps:

```
1. User Query
   ↓
2. Query Embedding (Convert query to vector)
   ↓
3. Vector Search (Find similar documents in vector store)
   ↓
4. Retrieve Top-K Documents
   ↓
5. Combine Query + Retrieved Documents
   ↓
6. LLM Generation (Generate response with context)
   ↓
7. Final Response
```

### Key Components of RAG

1. **Document Store** - A collection of documents (knowledge base)
2. **Embeddings** - Vector representations of documents and queries
3. **Vector Store** - Database that stores embeddings and enables similarity search
4. **Retriever** - Component that finds relevant documents for a query
5. **LLM** - Generates responses using the retrieved context

### Advantages of RAG

✅ **Up-to-date Information** - Can access information not in training data
✅ **Source Grounding** - Responses are grounded in retrieved documents
✅ **Scalability** - Can handle large knowledge bases
✅ **Flexibility** - Easy to update knowledge base without retraining
✅ **Reduced Hallucination** - Responses based on actual documents

### Limitations of RAG

❌ **Retrieval Latency** - Additional time for vector search
❌ **System Complexity** - Requires vector store, embeddings, retrieval logic
❌ **Retrieval Quality** - May retrieve irrelevant documents
❌ **Context Window Limits** - Can only include limited retrieved documents
❌ **Embedding Quality** - Quality depends on embedding model

### When to Use RAG

- Large, frequently updated knowledge bases
- Need for source citations
- Dynamic information requirements
- Domain-specific knowledge bases
- When retrieval quality is more important than latency

## 3. Implementing RAG

Let's implement a RAG system step by step. We'll create a knowledge base about AI agents and use it to answer questions.

### 3.1 Setting Up the Environment

First, let's import the necessary libraries:

In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# Load environment variables
load_dotenv()

print("✅ Environment setup complete!")

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


✅ Environment setup complete!


### 3.2 Creating a Knowledge Base

Let's create a simple knowledge base with documents about AI agents:

In [2]:
# Sample knowledge base documents
documents = [
    Document(
        page_content="LangGraph is a framework for building stateful, multi-actor applications with LLMs. It provides a graph-based approach to agent design, allowing developers to create complex workflows with nodes and edges. LangGraph supports both synchronous and asynchronous execution, making it suitable for production applications."
    ),
    Document(
        page_content="Retrieval-Augmented Generation (RAG) is a technique that enhances LLM responses by retrieving relevant documents from an external knowledge base. The process involves embedding queries and documents, performing vector similarity search, and including retrieved context in the LLM prompt. RAG reduces hallucinations and enables access to up-to-date information."
    ),
    Document(
        page_content="Tools in LangGraph allow agents to interact with external systems. Tools can be created using the @tool decorator or by extending BaseTool. Agents use tools by calling them through tool nodes in the graph. Tool execution results are returned to the agent for further processing."
    ),
    Document(
        page_content="Memory in LangGraph comes in two forms: short-term memory (conversation history) and long-term memory (persistent storage). Short-term memory is managed by checkpointers, while long-term memory uses stores with optional semantic search capabilities. Memory enables agents to maintain context across conversations."
    ),
    Document(
        page_content="The Model Context Protocol (MCP) is a standardized protocol for connecting AI agents to external services. MCP servers provide tools and resources that agents can discover and use. This enables agents to access real-time data, APIs, and services without custom integration code."
    ),
]

print(f"✅ Created knowledge base with {len(documents)} documents")
for i, doc in enumerate(documents, 1):
    print(f"   {i}. {doc.page_content[:80]}...")

✅ Created knowledge base with 5 documents
   1. LangGraph is a framework for building stateful, multi-actor applications with LL...
   2. Retrieval-Augmented Generation (RAG) is a technique that enhances LLM responses ...
   3. Tools in LangGraph allow agents to interact with external systems. Tools can be ...
   4. Memory in LangGraph comes in two forms: short-term memory (conversation history)...
   5. The Model Context Protocol (MCP) is a standardized protocol for connecting AI ag...


### 3.3 Splitting Documents

For better retrieval, we should split large documents into smaller chunks. This allows more precise retrieval:

In [3]:
# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,  # Characters per chunk
    chunk_overlap=50,  # Overlap between chunks
    length_function=len
)

chunks = text_splitter.split_documents(documents)

print(f"✅ Split {len(documents)} documents into {len(chunks)} chunks")
print(f"\nExample chunk:")
print(f"   {chunks[0].page_content[:150]}...")

✅ Split 5 documents into 11 chunks

Example chunk:
   LangGraph is a framework for building stateful, multi-actor applications with LLMs. It provides a graph-based approach to agent design, allowing devel...


### 3.4 Creating Embeddings and Vector Store

Now let's create embeddings and store them in a vector database:

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Initialize embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Create vector store (using ChromaDB)
# We'll use an in-memory store for this example
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="agent_knowledge_base"
)

print("✅ Vector store created with embeddings!")
print(f"   Stored {len(chunks)} document chunks")

✅ Vector store created with embeddings!
   Stored 11 document chunks


### 3.5 Creating a Retriever

The retriever finds relevant documents for a query:

In [5]:
# Create a retriever from the vector store
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Retrieve top 3 most similar documents
)

# Test the retriever
test_query = "What is LangGraph?"
retrieved_docs = retriever.invoke(test_query)

print(f"✅ Retriever created!")
print(f"\nQuery: '{test_query}'")
print(f"Retrieved {len(retrieved_docs)} documents:\n")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"{i}. {doc.page_content[:100]}...")

✅ Retriever created!

Query: 'What is LangGraph?'
Retrieved 3 documents:

1. to create complex workflows with nodes and edges. LangGraph supports both synchronous and asynchrono...
2. LangGraph is a framework for building stateful, multi-actor applications with LLMs. It provides a gr...
3. Memory in LangGraph comes in two forms: short-term memory (conversation history) and long-term memor...


### 3.6 Building the RAG Chain

Now let's create a RAG chain that combines retrieval with generation:

In [6]:
# Initialize LLM
llm = init_chat_model("gemini-2.0-flash-lite", model_provider="google_genai")

# Create a prompt template that includes retrieved context
prompt_template = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that answers questions based on the provided context.
    
Use the following retrieved documents to answer the question. If the context doesn't contain enough information to answer the question, say so.

Context:
{context}"""),
    ("human", "{question}"),
])

# Create the RAG chain
def format_docs(docs):
    """Format retrieved documents into a single string."""
    return "\n\n".join([doc.page_content for doc in docs])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("✅ RAG chain created!")

✅ RAG chain created!


### 3.7 Testing the RAG System

Let's test our RAG system with some questions:


In [7]:
# Test queries
queries = [
    "What is LangGraph?",
    "How does RAG work?",
    "What are tools in LangGraph?",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    response = rag_chain.invoke(query)
    print(f"Answer: {response}\n")


Query: What is LangGraph?
Answer: LangGraph is a framework for building stateful, multi-actor applications with LLMs. It provides a graph-based approach to agent design, allowing developers to create complex workflows with nodes and edges.


Query: How does RAG work?
Answer: RAG works by retrieving relevant documents from an external knowledge base to enhance LLM responses. The process involves embedding queries and documents, performing vector similarity search, and including retrieved context in the LLM prompt.


Query: What are tools in LangGraph?
Answer: Tools in LangGraph allow agents to interact with external systems. They can be created using the @tool decorator or by extending BaseTool.



### 3.8 Integrating RAG into an Agent

Now let's create a tool that uses RAG, which can be integrated into a LangGraph agent:


In [8]:
from langchain_core.tools import tool
from agentic_ai.agents.tool_agent import ToolAgent
from langgraph.checkpoint.memory import InMemorySaver

# Create a RAG tool
@tool
def search_knowledge_base(query: str) -> str:
    """Search the knowledge base for information about AI agents, LangGraph, RAG, tools, and memory.
    
    Use this tool when you need to find information from the knowledge base.
    
    Args:
        query: The search query
        
    Returns:
        Relevant information from the knowledge base
    """
    # Retrieve relevant documents
    docs = retriever.invoke(query)
    
    # Format and return
    if docs:
        return "\n\n".join([doc.page_content for doc in docs])
    else:
        return "No relevant information found."

# Create agent with RAG tool
memory = InMemorySaver()
tools = [search_knowledge_base]
agent = ToolAgent(
    llm=llm,
    tools=tools,
    name="RAGAgent",
    checkpointer=memory
)

print("✅ RAG agent created!")
print(f"   Available tools: {[tool.name for tool in tools]}")

✅ RAG agent created!
   Available tools: ['search_knowledge_base']


### 3.9 Testing the RAG Agent

Let's test the agent with questions that require knowledge base retrieval:


In [9]:
# Test the agent
test_question = "What is RAG and how does it work?"
print(f"Question: {test_question}\n")
print("="*60)

result = agent.run(test_question, thread_id="rag_test_001")
response = result['messages'][-1].content

print(f"Answer:\n{response}")

Question: What is RAG and how does it work?

Answer:
Retrieval-Augmented Generation (RAG) is a technique that improves Large Language Model (LLM) responses by fetching relevant documents from an external knowledge base. It works by embedding queries and documents, using vector similarity search to find the most relevant information, and then including this information in the LLM prompt. This helps reduce the chances of the LLM providing incorrect information (hallucinations) and allows it to access up-to-date information.


## 4. Cache-Augmented Generation (CAG)

**Cache-Augmented Generation (CAG)** is a technique that preloads relevant information directly into the LLM's context window, eliminating the need for real-time retrieval during inference.

### 4.1 How CAG Works

The CAG process follows these steps:

```
1. Knowledge Base (Pre-computed)
   ↓
2. Preload Relevant Information into Context
   ↓
3. User Query
   ↓
4. LLM Generation (with preloaded context)
   ↓
5. Final Response
```

### Key Components of CAG

1. **Knowledge Base** - Pre-computed collection of information
2. **Context Window** - The LLM's input context (e.g., 128K tokens)
3. **Preloading Strategy** - How to select and format information
4. **LLM** - Generates responses using preloaded context

### Advantages of CAG

✅ **Low Latency** - No retrieval step during inference
✅ **Simplified Architecture** - No vector store or retrieval logic needed
✅ **Reliability** - No retrieval errors or irrelevant documents
✅ **Consistent Context** - Same context for all queries in a session
✅ **Cost Efficiency** - Single LLM call instead of retrieval + generation

### Limitations of CAG

❌ **Context Window Limits** - Limited by model's context window size
❌ **Static Information** - Hard to update during runtime
❌ **Less Flexible** - Can't dynamically retrieve based on query
❌ **Memory Constraints** - Can't handle very large knowledge bases
❌ **Overhead** - May include irrelevant information in context

### When to Use CAG

- Small to medium knowledge bases that fit in context window
- Static or slowly changing information
- Low latency requirements
- Simple system architecture preferred
- When all queries benefit from the same context


## 5. Implementing CAG

Let's implement a CAG system that preloads context into the LLM prompt.

### 5.1 Preparing the Knowledge Base

First, let's prepare our knowledge base content:


In [10]:
# Prepare knowledge base content for CAG
cag_knowledge_base = """
# AI Agents Knowledge Base

## LangGraph
LangGraph is a framework for building stateful, multi-actor applications with LLMs. It provides a graph-based approach to agent design, allowing developers to create complex workflows with nodes and edges. LangGraph supports both synchronous and asynchronous execution, making it suitable for production applications.

## Retrieval-Augmented Generation (RAG)
Retrieval-Augmented Generation (RAG) is a technique that enhances LLM responses by retrieving relevant documents from an external knowledge base. The process involves embedding queries and documents, performing vector similarity search, and including retrieved context in the LLM prompt. RAG reduces hallucinations and enables access to up-to-date information.

## Tools in LangGraph
Tools in LangGraph allow agents to interact with external systems. Tools can be created using the @tool decorator or by extending BaseTool. Agents use tools by calling them through tool nodes in the graph. Tool execution results are returned to the agent for further processing.

## Memory in LangGraph
Memory in LangGraph comes in two forms: short-term memory (conversation history) and long-term memory (persistent storage). Short-term memory is managed by checkpointers, while long-term memory uses stores with optional semantic search capabilities. Memory enables agents to maintain context across conversations.

## Model Context Protocol (MCP)
The Model Context Protocol (MCP) is a standardized protocol for connecting AI agents to external services. MCP servers provide tools and resources that agents can discover and use. This enables agents to access real-time data, APIs, and services without custom integration code.
"""

print("✅ Knowledge base prepared for CAG")
print(f"   Knowledge base length: {len(cag_knowledge_base)} characters")


✅ Knowledge base prepared for CAG
   Knowledge base length: 1714 characters


### 5.2 Creating a CAG Prompt Template

We'll create a prompt that includes the preloaded knowledge base:


In [11]:
# Create CAG prompt template with preloaded context
cag_prompt_template = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant with access to a knowledge base about AI agents.

Use the following knowledge base to answer questions. The knowledge base contains information about LangGraph, RAG, tools, memory, and MCP.

Knowledge Base:
{knowledge_base}

When answering questions:
1. Use information from the knowledge base when relevant
2. If the knowledge base doesn't contain relevant information, say so
3. Be concise and accurate
"""),
    ("human", "{question}"),
])

print("✅ CAG prompt template created!")


✅ CAG prompt template created!


### 5.3 Building the CAG Chain

Now let's create a CAG chain that uses the preloaded context:


In [12]:
# Create CAG chain
cag_chain = (
    {"knowledge_base": lambda x: cag_knowledge_base, "question": RunnablePassthrough()}
    | cag_prompt_template
    | llm
    | StrOutputParser()
)

print("✅ CAG chain created!")

✅ CAG chain created!


### 5.4 Testing the CAG System

Let's test our CAG system:


In [13]:
# Test queries
queries = [
    "What is LangGraph?",
    "How does RAG work?",
    "What are tools in LangGraph?",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    response = cag_chain.invoke(query)
    print(f"Answer: {response}\n")



Query: What is LangGraph?
Answer: LangGraph is a framework for building stateful, multi-actor applications with LLMs. It uses a graph-based approach to agent design, allowing developers to create complex workflows with nodes and edges. It supports both synchronous and asynchronous execution.


Query: How does RAG work?
Answer: Retrieval-Augmented Generation (RAG) enhances LLM responses by retrieving relevant documents from an external knowledge base. The process involves embedding queries and documents, performing vector similarity search, and including retrieved context in the LLM prompt. This reduces hallucinations and enables access to up-to-date information.


Query: What are tools in LangGraph?
Answer: Tools in LangGraph allow agents to interact with external systems. They can be created using the @tool decorator or by extending BaseTool. Agents use tools by calling them through tool nodes in the graph, and the execution results are returned to the agent for further processing.



### 5.5 Creating a CAG Agent

Let's create an agent that uses CAG by including the knowledge base in the system prompt:


In [14]:
from agentic_ai.agents.simple_agent import SimpleAgent

# Create system prompt with preloaded knowledge base
cag_system_prompt = f"""You are a helpful assistant with access to a knowledge base about AI agents.

Knowledge Base:
{cag_knowledge_base}

When answering questions:
1. Use information from the knowledge base when relevant
2. If the knowledge base doesn't contain relevant information, say so
3. Be concise and accurate
"""

# Create CAG agent
cag_agent = SimpleAgent(
    llm=llm,
    name="CAGAgent",
    system_prompt=cag_system_prompt,
    checkpointer=memory
)

print("✅ CAG agent created!")


✅ CAG agent created!


### 5.6 Testing the CAG Agent

Let's test the CAG agent:


In [15]:
# Test the agent
test_question = "What is RAG and how does it work?"
print(f"Question: {test_question}\n")
print("="*60)

result = cag_agent.run(test_question, thread_id="cag_test_001")
response = result['messages'][-1].content

print(f"Answer:\n{response}")


Question: What is RAG and how does it work?

Answer:
Retrieval-Augmented Generation (RAG) is a technique that enhances LLM responses by retrieving relevant documents from an external knowledge base. The process involves embedding queries and documents, performing vector similarity search, and including retrieved context in the LLM prompt.


## 6. RAG vs CAG: Comparison

Let's compare RAG and CAG across different dimensions:

### 6.1 Comparison Table

| Aspect | RAG | CAG |
|--------|-----|-----|
| **Mechanism** | Retrieves documents during inference | Preloads context before inference |
| **Latency** | Higher (retrieval + generation) | Lower (generation only) |
| **Architecture** | Complex (vector store, embeddings, retriever) | Simple (just prompt with context) |
| **Knowledge Base Size** | Can handle very large KBs | Limited by context window |
| **Update Frequency** | Easy to update dynamically | Requires rebuilding context |
| **Query-Specific Context** | Yes, retrieves based on query | No, same context for all queries |
| **Retrieval Quality** | Depends on embedding/retrieval quality | No retrieval, so no retrieval errors |
| **Context Efficiency** | Only includes relevant documents | May include irrelevant information |
| **Cost** | Embedding + retrieval + generation | Generation only |
| **Best For** | Large, dynamic knowledge bases | Small, static knowledge bases |

### 6.2 When to Use Each

**Use RAG when:**
- Knowledge base is large (doesn't fit in context window)
- Information changes frequently
- Different queries need different context
- You need source citations
- Retrieval quality is more important than latency

**Use CAG when:**
- Knowledge base fits in context window
- Information is static or changes slowly
- Low latency is critical
- Simple architecture is preferred
- All queries benefit from the same context

### 6.3 Hybrid Approaches

You can also combine RAG and CAG:

1. **CAG for Core Knowledge** - Preload frequently used, static information
2. **RAG for Dynamic Information** - Retrieve query-specific, frequently updated information
3. **Layered Context** - Use CAG for base context, RAG for specific queries


## 7. Performance Comparison

Let's compare the performance of RAG and CAG on the same queries:


In [16]:
import time

# Test queries
test_queries = [
    "What is LangGraph?",
    "How does RAG work?",
    "What are tools in LangGraph?",
]

print("Performance Comparison: RAG vs CAG\n")
print("="*60)

for query in test_queries:
    print(f"\nQuery: {query}")
    print("-"*60)
    
    # Test RAG
    start_time = time.time()
    rag_response = rag_chain.invoke(query)
    rag_time = time.time() - start_time
    
    # Test CAG
    start_time = time.time()
    cag_response = cag_chain.invoke(query)
    cag_time = time.time() - start_time
    
    print(f"RAG Time: {rag_time:.3f}s")
    print(f"CAG Time: {cag_time:.3f}s")
    print(f"Speedup: {rag_time/cag_time:.2f}x faster with CAG")
    
    # Show response lengths
    print(f"\nRAG Response Length: {len(rag_response)} chars")
    print(f"CAG Response Length: {len(cag_response)} chars")


Performance Comparison: RAG vs CAG


Query: What is LangGraph?
------------------------------------------------------------
RAG Time: 5.396s
CAG Time: 5.131s
Speedup: 1.05x faster with CAG

RAG Response Length: 205 chars
CAG Response Length: 201 chars

Query: How does RAG work?
------------------------------------------------------------
RAG Time: 2.618s
CAG Time: 0.895s
Speedup: 2.93x faster with CAG

RAG Response Length: 242 chars
CAG Response Length: 340 chars

Query: What are tools in LangGraph?
------------------------------------------------------------
RAG Time: 1.596s
CAG Time: 1.122s
Speedup: 1.42x faster with CAG

RAG Response Length: 138 chars
CAG Response Length: 278 chars


## 8. Advanced RAG Techniques

Let's explore some advanced RAG techniques:

### 8.1 Query Expansion

Query expansion improves retrieval by generating multiple query variations:


In [17]:
# Simple query expansion example
def expand_query(query: str) -> list[str]:
    """Generate query variations."""
    expansions = [
        query,
        f"What is {query}?",
        f"Explain {query}",
        f"Tell me about {query}",
    ]
    return expansions

# Test query expansion
original_query = "LangGraph"
expanded_queries = expand_query(original_query)

print("Query Expansion Example:")
print(f"Original: {original_query}")
print(f"Expanded: {expanded_queries}")

# Retrieve with expanded queries
all_docs = []
for eq in expanded_queries:
    docs = retriever.invoke(eq)
    all_docs.extend(docs)

# Remove duplicates (by content)
unique_docs = []
seen_content = set()
for doc in all_docs:
    if doc.page_content not in seen_content:
        unique_docs.append(doc)
        seen_content.add(doc.page_content)

print(f"\nRetrieved {len(unique_docs)} unique documents with query expansion")


Query Expansion Example:
Original: LangGraph
Expanded: ['LangGraph', 'What is LangGraph?', 'Explain LangGraph', 'Tell me about LangGraph']

Retrieved 4 unique documents with query expansion


### 8.2 Re-ranking

Re-ranking improves retrieval quality by re-scoring retrieved documents:


In [18]:
# Simple re-ranking by keyword matching
def rerank_documents(query: str, docs: list[Document], top_k: int = 3) -> list[Document]:
    """Re-rank documents based on keyword overlap."""
    query_words = set(query.lower().split())
    
    scored_docs = []
    for doc in docs:
        doc_words = set(doc.page_content.lower().split())
        overlap = len(query_words & doc_words)
        scored_docs.append((overlap, doc))
    
    # Sort by score (descending) and return top_k
    scored_docs.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored_docs[:top_k]]

# Test re-ranking
query = "LangGraph framework"
docs = retriever.invoke(query)
reranked = rerank_documents(query, docs, top_k=2)

print("Re-ranking Example:")
print(f"Query: {query}")
print(f"\nOriginal retrieval: {len(docs)} documents")
print(f"After re-ranking (top 2): {len(reranked)} documents")
for i, doc in enumerate(reranked, 1):
    print(f"\n{i}. {doc.page_content[:100]}...")


Re-ranking Example:
Query: LangGraph framework

Original retrieval: 3 documents
After re-ranking (top 2): 2 documents

1. LangGraph is a framework for building stateful, multi-actor applications with LLMs. It provides a gr...

2. to create complex workflows with nodes and edges. LangGraph supports both synchronous and asynchrono...


### 8.3 Hybrid Search

Combining semantic search with keyword search can improve retrieval:


In [19]:
# Simple hybrid search (semantic + keyword)
def hybrid_search(query: str, vectorstore, top_k: int = 3) -> list[Document]:
    """Combine semantic and keyword search."""
    # Semantic search
    semantic_docs = vectorstore.similarity_search(query, k=top_k)
    
    # Keyword search (simple implementation)
    query_words = set(query.lower().split())
    keyword_docs = []
    for doc in chunks:
        doc_words = set(doc.page_content.lower().split())
        if query_words & doc_words:  # If any overlap
            keyword_docs.append(doc)
    
    # Combine and deduplicate
    all_docs = semantic_docs + keyword_docs
    unique_docs = []
    seen_content = set()
    for doc in all_docs:
        if doc.page_content not in seen_content:
            unique_docs.append(doc)
            seen_content.add(doc.page_content)
    
    return unique_docs[:top_k]

# Test hybrid search
query = "LangGraph"
hybrid_results = hybrid_search(query, vectorstore, top_k=3)

print("Hybrid Search Example:")
print(f"Query: {query}")
print(f"Retrieved {len(hybrid_results)} documents")
for i, doc in enumerate(hybrid_results, 1):
    print(f"\n{i}. {doc.page_content[:100]}...")


Hybrid Search Example:
Query: LangGraph
Retrieved 3 documents

1. to create complex workflows with nodes and edges. LangGraph supports both synchronous and asynchrono...

2. LangGraph is a framework for building stateful, multi-actor applications with LLMs. It provides a gr...

3. Memory in LangGraph comes in two forms: short-term memory (conversation history) and long-term memor...


## 9. Exercises

Now it's time to practice! Complete the following exercises to master RAG and CAG:

### Exercise 1: Build a RAG System for a Custom Domain

Create a RAG system for a domain of your choice (e.g., cooking recipes, programming languages, sports):

1. Create a knowledge base with at least 5 documents about your chosen domain
2. Set up embeddings and a vector store
3. Create a retriever that returns the top 3 most relevant documents
4. Build a RAG chain that answers questions using retrieved context
5. Test with at least 3 different queries

**Hints:**
- Use `RecursiveCharacterTextSplitter` to split documents
- Use `Chroma.from_documents()` to create the vector store
- Create a prompt template that includes retrieved context


In [20]:
# TODO: Complete Exercise 1
# from langchain_core.documents import Document
# from langchain_chroma import Chroma
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# 
# # 1. Create your knowledge base
# # TODO: Create documents for your chosen domain
# 
# # 2. Split documents
# # TODO: Use RecursiveCharacterTextSplitter
# 
# # 3. Create vector store
# # TODO: Use Chroma.from_documents()
# 
# # 4. Create retriever
# # TODO: Create retriever with k=3
# 
# # 5. Build RAG chain
# # TODO: Create prompt template and chain
# 
# # 6. Test with queries
# # TODO: Test with at least 3 queries


### Exercise 2: Implement CAG for a FAQ System

Create a CAG system for a FAQ (Frequently Asked Questions) system:

1. Create a FAQ knowledge base with at least 10 Q&A pairs
2. Format the FAQ as a preloaded context string
3. Create a CAG prompt template that includes the FAQ
4. Build a CAG chain that answers questions using the preloaded FAQ
5. Test with questions that match and don't match the FAQ

**Hints:**
- Format FAQ as markdown or structured text
- Include instructions for the LLM on how to use the FAQ
- Test with both matching and non-matching queries


In [21]:
# TODO: Complete Exercise 2
# 
# # 1. Create FAQ knowledge base
# # TODO: Create at least 10 Q&A pairs
# 
# # 2. Format as preloaded context
# # TODO: Format FAQ as a string
# 
# # 3. Create CAG prompt template
# # TODO: Create prompt with FAQ context
# 
# # 4. Build CAG chain
# # TODO: Create chain with preloaded FAQ
# 
# # 5. Test with queries
# # TODO: Test with matching and non-matching queries


### Exercise 3: Compare RAG vs CAG Performance

Compare RAG and CAG on the same knowledge base:

1. Use the same knowledge base for both RAG and CAG
2. Create test queries (at least 5)
3. Measure response time for both approaches
4. Compare response quality (you can manually evaluate)
5. Document your findings

**Hints:**
- Use `time.time()` to measure latency
- Test with queries of varying complexity
- Consider both speed and quality in your comparison


In [22]:
# TODO: Complete Exercise 3
# import time
# 
# # 1. Use same knowledge base
# # TODO: Create knowledge base
# 
# # 2. Create test queries
# # TODO: Create at least 5 test queries
# 
# # 3. Measure RAG performance
# # TODO: Measure time and quality for RAG
# 
# # 4. Measure CAG performance
# # TODO: Measure time and quality for CAG
# 
# # 5. Compare and document
# # TODO: Create comparison table and document findings


### Exercise 4: Create a Hybrid RAG-CAG Agent

Create an agent that combines RAG and CAG:

1. Use CAG for core, static knowledge (preloaded in system prompt)
2. Use RAG for dynamic, query-specific information
3. Create an agent that can use both approaches
4. Test with queries that benefit from each approach

**Hints:**
- Preload frequently used information in the system prompt
- Use RAG tool for query-specific retrieval
- Design prompts that guide when to use each approach


In [23]:
# TODO: Complete Exercise 4
# from agentic_ai.agents.tool_agent import ToolAgent
# 
# # 1. Create CAG knowledge base (core, static)
# # TODO: Create core knowledge base
# 
# # 2. Create RAG system (dynamic, query-specific)
# # TODO: Create RAG vector store and retriever
# 
# # 3. Create RAG tool
# # TODO: Create tool that uses RAG retriever
# 
# # 4. Create hybrid agent
# # TODO: Create agent with CAG system prompt + RAG tool
# 
# # 5. Test with different query types
# # TODO: Test queries that benefit from CAG and RAG


### Exercise 5: Advanced RAG with Re-ranking

Implement an advanced RAG system with re-ranking:

1. Create a RAG system with a larger knowledge base (10+ documents)
2. Implement a re-ranking function that scores documents
3. Retrieve more documents initially (e.g., top 10)
4. Re-rank and select top 3 for final context
5. Compare results with and without re-ranking

**Hints:**
- Use keyword overlap, semantic similarity, or both for re-ranking
- Test with queries where re-ranking makes a difference
- Consider implementing a more sophisticated scoring function


In [24]:
# TODO: Complete Exercise 5
# 
# # 1. Create larger knowledge base
# # TODO: Create 10+ documents
# 
# # 2. Implement re-ranking function
# # TODO: Create function that scores documents
# 
# # 3. Create RAG with re-ranking
# # TODO: Retrieve top 10, re-rank, select top 3
# 
# # 4. Compare with baseline
# # TODO: Compare results with and without re-ranking
# 
# # 5. Evaluate improvement
# # TODO: Document if re-ranking improves results


## 10. Summary: Augmented Generation (RAG & CAG)

**Key Takeaways:**

1. **Augmented Generation enhances LLMs** by providing additional context beyond training data
   - Enables access to up-to-date information
   - Reduces hallucinations
   - Improves accuracy and grounding

2. **RAG (Retrieval-Augmented Generation)**:
   - Retrieves relevant documents during inference
   - Uses vector stores and embeddings for similarity search
   - Best for large, dynamic knowledge bases
   - More complex architecture but more flexible

3. **CAG (Cache-Augmented Generation)**:
   - Preloads context into the LLM's input
   - Simpler architecture, lower latency
   - Best for small, static knowledge bases
   - Limited by context window size

4. **Key Differences**:
   - **Latency**: CAG is faster (no retrieval step)
   - **Flexibility**: RAG is more flexible (query-specific retrieval)
   - **Scalability**: RAG scales better (handles large KBs)
   - **Complexity**: CAG is simpler (no vector store needed)

5. **When to Use Each**:
   - **Use RAG** for large KBs, dynamic information, need for citations
   - **Use CAG** for small KBs, static information, low latency needs
   - **Use Hybrid** for combining core (CAG) + dynamic (RAG) knowledge

6. **Implementation Pattern**:
   ```python
   # RAG Pattern
   retriever = vectorstore.as_retriever()
   rag_chain = (
       {"context": retriever | format_docs, "question": RunnablePassthrough()}
       | prompt_template
       | llm
   )
   
   # CAG Pattern
   cag_chain = (
       {"context": lambda x: preloaded_kb, "question": RunnablePassthrough()}
       | prompt_template
       | llm
   )
   ```

7. **Advanced Techniques**:
   - Query expansion for better retrieval
   - Re-ranking for improved quality
   - Hybrid search (semantic + keyword)
   - Hybrid RAG-CAG systems

**Next Steps:**
- Experiment with different embedding models
- Try different vector stores (Chroma, FAISS, Pinecone)
- Implement more sophisticated re-ranking algorithms
- Explore agentic RAG (where agents decide when to retrieve)
- Learn about evaluation metrics for RAG systems

**Resources:**
- LangChain RAG Documentation: https://python.langchain.com/docs/use_cases/question_answering/
- LangGraph Agentic RAG: https://langchain-ai.github.io/langgraph/tutorials/rag/langgraph_agentic_rag/
- ChromaDB Documentation: https://docs.trychroma.com/
- Embeddings Guide: https://python.langchain.com/docs/integrations/text_embedding/
